# Search App-33 — Rush Hour : l'A* admissible sur un puzzle à glissement

**Navigation** : [App-34 — 15-puzzle A*](App-34-15Puzzle-Admissible-Astar-CSharp.ipynb) | [Index](../README.md) | [Version Python](App-33-RushHour-Admissible-Astar-Python.ipynb)

Le puzzle comporte une grille 6 × 6 et une voiture rouge à amener à la sortie en haut à droite. Nous construisons ses états et comparons BFS, A* et une recherche gloutonne sans retour arrière.

**Objectifs** : modéliser les glissements, justifier une heuristique admissible et comparer états développés et nombre de coups. **Prérequis** : BFS, A* et files de priorité. **Durée** : 30 minutes. La BCL suffit : les algorithmes sont l'objet de l'étude.

## 1. États, déplacements et but

Chaque voiture de longueur deux est `(ligne, colonne, orientation, rouge)` ; la coordonnée est sa case gauche ou haute. Un glissement de n'importe quelle distance libre compte pour **un coup**. Le but place la rouge horizontale en `(0,4)`, couvrant les deux cases à droite de la première ligne. Les clés trient les voitures pour ne pas dépendre de leur ordre dans une liste.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
CultureInfo.DefaultThreadCurrentCulture=CultureInfo.InvariantCulture;
Console.WriteLine($"Runtime {System.Runtime.InteropServices.RuntimeInformation.FrameworkDescription} ; BCL");

Runtime .NET 10.0.14 ; BCL


Le moteur ci-dessous énumère tous les glissements dans les deux sens et trie les clés. Un état ne contient ni chevauchement ni voiture hors grille.

In [2]:
#nullable enable
record struct Voiture(int R,int C,char O,bool Rouge);
record Recherche(List<string>? Chemin,int Developpes);
static class Rush {
 public const int N=6;
 public static (int r,int c)[] Cases(Voiture v)=>v.O=='H'?new[]{(v.R,v.C),(v.R,v.C+1)}:new[]{(v.R,v.C),(v.R+1,v.C)};
 public static HashSet<(int,int)> Occupe(IEnumerable<Voiture> vs)=>vs.SelectMany(Cases).ToHashSet();
 public static string Cle(IEnumerable<Voiture> vs)=>string.Concat(vs.Select(v=>$"{v.R}{v.C}{v.O}{(v.Rouge?1:0)}").OrderBy(s=>s,StringComparer.Ordinal));
 public static Voiture[] Lire(string s)=>Enumerable.Range(0,s.Length/4).Select(i=>new Voiture(s[4*i]-'0',s[4*i+1]-'0',s[4*i+2],s[4*i+3]=='1')).ToArray();
 public static bool But(string s)=>Lire(s).Any(v=>v.Rouge&&v.R==0&&v.C==4&&v.O=='H');
 public static List<Voiture> Glissements(Voiture v,HashSet<(int,int)> occ){
  var result=new List<Voiture>();
  if(v.O=='H') {for(int k=1;v.C+k+1<N&&!occ.Contains((v.R,v.C+k+1));k++)result.Add(v with {C=v.C+k});for(int k=1;v.C-k>=0&&!occ.Contains((v.R,v.C-k));k++)result.Add(v with {C=v.C-k});}
  else {for(int k=1;v.R+k+1<N&&!occ.Contains((v.R+k+1,v.C));k++)result.Add(v with {R=v.R+k});for(int k=1;v.R-k>=0&&!occ.Contains((v.R-k,v.C));k++)result.Add(v with {R=v.R-k});}
  return result;
 }
 public static List<string> Voisins(string s){var vs=Lire(s);var occ=Occupe(vs);var result=new SortedSet<string>(StringComparer.Ordinal);for(int i=0;i<vs.Length;i++)foreach(var v in Glissements(vs[i],occ)){var x=(Voiture[])vs.Clone();x[i]=v;result.Add(Cle(x));}return result.ToList();}
 public static int H(string s){var vs=Lire(s);var rouge=vs.Single(v=>v.Rouge);if(rouge.R!=0)return 0;return vs.Count(v=>!v.Rouge&&Cases(v).Any(p=>p.r==0&&p.c>rouge.C+1));}
 public static string Afficher(string s){var grille=new char[N,N];for(int r=0;r<N;r++)for(int c=0;c<N;c++)grille[r,c]='.';foreach(var v in Lire(s))foreach(var p in Cases(v))grille[p.r,p.c]=v.Rouge?'R':v.O=='H'?'-':'|';return string.Join("\n",Enumerable.Range(0,N).Select(r=>string.Join("  ",Enumerable.Range(0,N).Select(c=>grille[r,c]))+(r==0?"  <== SORTIE":"")));}
 public static string Base()=>Cle(new[]{new Voiture(0,0,'H',true),new Voiture(0,2,'V',false),new Voiture(0,4,'V',false),new Voiture(3,1,'H',false),new Voiture(4,3,'H',false),new Voiture(2,5,'V',false)});
 public static string Melanger(string s,int steps,int seed){var rng=new Random(seed);for(int i=0;i<steps;i++){var ns=Voisins(s);s=ns[rng.Next(ns.Count)];}return s;}
 public static List<string> Chemin(Dictionary<string,string?> parent,string s){var p=new List<string>{s};while(parent[s] is string prev){s=prev;p.Add(s);}p.Reverse();return p;}
 public static Recherche BFS(string debut){var q=new Queue<string>();q.Enqueue(debut);var parent=new Dictionary<string,string?>{{debut,null}};int n=0;while(q.Count>0){var s=q.Dequeue();n++;if(But(s))return new(Chemin(parent,s),n);foreach(var v in Voisins(s))if(!parent.ContainsKey(v)){parent[v]=s;q.Enqueue(v);}}return new(null,n);}
 public static Recherche AStar(string debut){var q=new PriorityQueue<(string s,int g),(int f,int g,string s)>();q.Enqueue((debut,0),(H(debut),0,debut));var dist=new Dictionary<string,int>{{debut,0}};var parent=new Dictionary<string,string?>{{debut,null}};var closed=new HashSet<string>();int n=0;while(q.TryDequeue(out var item,out _)){var(s,g)=item;if(!closed.Add(s))continue;n++;if(But(s))return new(Chemin(parent,s),n);foreach(var v in Voisins(s)){int ng=g+1;if(!dist.TryGetValue(v,out int old)||ng<old){dist[v]=ng;parent[v]=s;q.Enqueue((v,ng),(ng+H(v),ng,v));}}}return new(null,n);}
 public static Recherche Glouton(string debut){string s=debut;var seen=new HashSet<string>{s};var p=new List<string>{s};int n=0;while(true){n++;if(But(s))return new(p,n);var next=Voisins(s).Where(v=>!seen.Contains(v)).OrderBy(H).FirstOrDefault();if(next is null)return new(null,n);s=next;seen.Add(s);p.Add(s);}}
 public static bool Valide(string s){var vs=Lire(s);if(vs.Count(v=>v.Rouge)!=1 || vs.Any(v=>v.O!='H'&&v.O!='V'))return false;var pts=vs.SelectMany(Cases).ToArray();return pts.All(p=>p.r>=0&&p.r<N&&p.c>=0&&p.c<N)&&pts.Distinct().Count()==pts.Length;}
 // Oracle de transition independant : une voiture change, toutes ses cases balayees sont libres.
 public static bool Transition(string avant,string apres){if(!Valide(avant)||!Valide(apres))return false;var a=Lire(avant).ToList();var b=Lire(apres).ToList();foreach(var v in a.ToArray())if(b.Remove(v))a.Remove(v);if(a.Count!=1||b.Count!=1)return false;var x=a[0];var y=b[0];if(x.O!=y.O||x.Rouge!=y.Rouge)return false;int dr=y.R-x.R,dc=y.C-x.C;if(x.O=='H'?(dr!=0||dc==0):(dc!=0||dr==0))return false;var reste=Lire(avant).ToList();reste.Remove(x);var occ=Occupe(reste);int steps=Math.Abs(dr)+Math.Abs(dc);for(int i=1;i<=steps;i++){var z=x with {R=x.R+i*Math.Sign(dr),C=x.C+i*Math.Sign(dc)};if(Cases(z).Any(p=>p.r<0||p.r>=N||p.c<0||p.c>=N||occ.Contains(p)))return false;}return true;}
 public static bool CheminValide(string debut,Recherche r)=>r.Chemin is not null&&r.Chemin[0]==debut&&But(r.Chemin[^1])&&r.Chemin.All(Valide)&&r.Chemin.Zip(r.Chemin.Skip(1),Transition).All(x=>x);
}
Console.WriteLine("Primitives et recherches initialisees : grille 6 x 6 ; un glissement = un coup");

Primitives et recherches initialisees : grille 6 x 6 ; un glissement = un coup


### Exemple guidé : lire une grille

`R` désigne la rouge, `-` une horizontale, `|` une verticale et `.` une case vide. La clé est indépendante de l'ordre des voitures.

In [3]:
var demo=Rush.Cle(new[]{new Voiture(0,2,'H',true),new Voiture(2,0,'V',false),new Voiture(4,3,'H',false)});
Console.WriteLine(Rush.Afficher(demo));Console.WriteLine($"voitures={Rush.Lire(demo).Length} ; cle={demo} ; voisins={Rush.Voisins(demo).Count}");

.  .  R  R  .  .  <== SORTIE
.  .  .  .  .  .
|  .  .  .  .  .
|  .  .  .  .  .
.  .  .  -  -  .
.  .  .  .  .  .


voitures=3 ; cle=02H120V043H0 ; voisins=12


### Exercice 1 — test de but

Compléter le stub pour reconnaître exactement une voiture rouge horizontale à gauche `(0,4)`. Les fixtures comportent un but et plusieurs faux buts. Les cas incluent plusieurs voitures, ordre permuté et absence de rouge.

In [4]:
static bool? ButExercice(Voiture[] voitures){
 // TODO : tester orientation, ligne et colonne de la voiture rouge.
 return null;
}
var b1=ButExercice(new[]{new Voiture(0,4,'H',true)});var b2=ButExercice(new[]{new Voiture(0,3,'H',true)});var b3=ButExercice(new[]{new Voiture(1,4,'H',true)});var b4=ButExercice(new[]{new Voiture(0,4,'V',true)});
var rougeAilleurs=new Voiture(1,0,'H',true);var autreSortie=new Voiture(0,4,'H',false);
var b5=ButExercice(new[]{autreSortie,rougeAilleurs});var b6=ButExercice(new[]{rougeAilleurs,autreSortie});
var rougeSortie=new Voiture(0,4,'H',true);var autre=new Voiture(1,0,'H',false);
var b7=ButExercice(new[]{rougeSortie,autre});var b8=ButExercice(new[]{autre,rougeSortie});var b9=ButExercice(new[]{autreSortie});
if(b1 is null||b2 is null||b3 is null||b4 is null||b5 is null||b6 is null||b7 is null||b8 is null||b9 is null)Console.WriteLine("Exercice 1 a completer");
else if(b1!=true||b2!=false||b3!=false||b4!=false||b5!=false||b6!=false||b7!=true||b8!=true||b9!=false)throw new Exception("test de but incorrect");else Console.WriteLine("Exercice 1 : neuf fixtures valides");

Exercice 1 a completer


## 2. Position de base et variantes

La position de base contient six voitures : la rouge en haut à gauche et deux verticales qui barrent sa ligne. Elle se résout en trois coups. Le scramble suit des glissements légaux depuis cette position ; ces déplacements étant réversibles, les variantes restent résolubles.

In [5]:
var instance=Rush.Base(); Console.WriteLine("Position de base :");Console.WriteLine(Rush.Afficher(instance));Console.WriteLine($"voitures={Rush.Lire(instance).Length} ; etat valide={Rush.Valide(instance)}");

Position de base :


R  R  |  .  |  .  <== SORTIE
.  .  |  .  |  .
.  .  .  .  .  |
.  -  -  .  .  |
.  .  .  -  -  .
.  .  .  .  .  .


voitures=6 ; etat valide=True


## 3. BFS : optimum en coups

BFS explore les états par niveaux. Sur un graphe où chaque glissement coûte un coup, le premier but atteint fournit un chemin de longueur minimale. Le nombre d'états développés mesure ici le travail de recherche, pas le temps machine.

In [6]:
var bfs=Rush.BFS(instance);if(!Rush.CheminValide(instance,bfs))throw new Exception("chemin BFS invalide");
Console.WriteLine($"BFS : {bfs.Chemin!.Count-1} coups, {bfs.Developpes} etats developpes ; transitions valides={bfs.Chemin.Count-1}");

BFS : 3 coups, 433 etats developpes ; transitions valides=3


La grille affichée et le chemin validé relient les règles du jeu à l'optimum. L'oracle de transition vérifie indépendamment les cases traversées, les bornes et les chevauchements.

Sur la position de base, BFS trouve 3 coups en développant 433 états ; les trois transitions sont vérifiées.

## 4. A* et bloqueurs

L'heuristique compte les voitures distinctes qui occupent la première ligne à droite de la rouge. Chacune doit être déplacée au moins une fois pour libérer la sortie : ce compte est une borne inférieure. Un seul glissement change le nombre de bloqueurs d'au plus un, ce qui motive la cohérence. A* combine coût déjà parcouru et borne restante ; les égalités sont départagées par coût puis clé.

In [7]:
var astar=Rush.AStar(instance);if(!Rush.CheminValide(instance,astar)||astar.Chemin!.Count!=bfs.Chemin!.Count)throw new Exception("A* ou optimum");
Console.WriteLine($"A* : {astar.Chemin.Count-1} coups, {astar.Developpes} etats ; rapport BFS/A*={(double)bfs.Developpes/astar.Developpes:F2}");
// Sur 40 etats atteignables distincts : BFS est un oracle de distance ;
// verifier la coherence sur tous leurs arcs sortants et les chemins examines.
var explore=new Queue<string>();var vus=new HashSet<string>{instance};explore.Enqueue(instance);int etats=0,arcs=0;
while(explore.Count>0 && etats<40){var s=explore.Dequeue();var refB=Rush.BFS(s);if(!Rush.CheminValide(s,refB)||Rush.H(s)>refB.Chemin!.Count-1)throw new Exception("admissibilite");foreach(var v in Rush.Voisins(s)){arcs++;if(!Rush.Transition(s,v)||Rush.H(s)>1+Rush.H(v))throw new Exception("coherence ou mouvement");if(vus.Add(v))explore.Enqueue(v);}etats++;}
Console.WriteLine($"oracle borne : {etats} etats, {arcs} arcs verifies ; 0 violation admissibilite/coherence/transitions");

A* : 3 coups, 100 etats ; rapport BFS/A*=4.33


oracle borne : 40 etats, 620 arcs verifies ; 0 violation admissibilite/coherence/transitions


La comparaison des longueurs vérifie l'optimalité sur la position de base. Le contrôle borné examine 40 états accessibles et tous leurs arcs sortants : il illustre admissibilité et cohérence sans prétendre avoir exploré tout l'espace.

A* trouve également 3 coups en développant 100 états, contre 433 pour BFS. Le contrôle explore 40 états accessibles et 620 arcs.

### Exercice 2 — compter les bloqueurs

Écrire l'heuristique à partir des cellules occupées. Les fixtures distinguent deux bloqueurs, un seul et aucun : une valeur constante ne suffit pas. Une voiture horizontale peut occuper deux cases de la ligne et ne compte pourtant qu’une fois.

In [8]:
static int? BloqueursExercice(Voiture[] voitures){
 // TODO : compter les voitures distinctes sur la ligne 0 a droite de la rouge.
 return null;
}
var h2=BloqueursExercice(Rush.Lire(instance));var h0=BloqueursExercice(new[]{new Voiture(0,2,'H',true)});var h1=BloqueursExercice(new[]{new Voiture(0,0,'H',true),new Voiture(0,3,'V',false)});
var rouge=new Voiture(0,0,'H',true);var horizontal=new Voiture(0,3,'H',false);var vertical=new Voiture(0,5,'V',false);
var hHorizontal=BloqueursExercice(new[]{rouge,horizontal});var hMixte=BloqueursExercice(new[]{vertical,rouge,horizontal});
if(h2 is null||h0 is null||h1 is null||hHorizontal is null||hMixte is null)Console.WriteLine("Exercice 2 a completer");else if(h2!=2||h0!=0||h1!=1||hHorizontal!=1||hMixte!=2)throw new Exception("bloqueurs incorrects");else Console.WriteLine("Exercice 2 : cinq fixtures valides");

Exercice 2 a completer


## 5. Glouton sans retour arrière

À chaque étape, le glouton choisit un voisin jamais visité de plus faible heuristique et ne revient pas sur ses pas. Il peut trouver une route plus longue ou rencontrer un cul-de-sac ; aucun de ces résultats n'est garanti sur toute instance.

In [9]:
var glouton=Rush.Glouton(instance);if(glouton.Chemin is not null&&!Rush.CheminValide(instance,glouton))throw new Exception("chemin glouton invalide");
Console.WriteLine(glouton.Chemin is null?$"glouton : cul-de-sac apres {glouton.Developpes} etats":$"glouton : {glouton.Chemin.Count-1} coups, {glouton.Developpes} etats ; optimum={bfs.Chemin!.Count-1}");

glouton : 134 coups, 135 etats ; optimum=3


La sortie compare ce chemin à l'optimum de BFS. La recherche gloutonne n'a pas de garantie d'optimalité ; son coût observé et le nombre d'états dépendent de l'instance et du départage.

Sur la position de base, le glouton fournit 134 coups et développe 135 états ; l’optimum est 3 coups.

### Exercice 3 — toutes les positions d'un glissement

Retourner chaque position légale d'une voiture pour les deux sens ; les cases de la voiture elle-même sont incluses dans `occ`. Les deux fixtures couvrent orientations, bords et obstacles et exigent les ensembles exacts.

In [10]:
#nullable enable
static List<Voiture>? GlissementsExercice(Voiture v,HashSet<(int,int)> occ){
 // TODO : parcourir les deux sens de l'axe et accumuler toutes les positions libres.
 return null;
}
var hv=new Voiture(0,2,'H',true);var vv=new Voiture(2,3,'V',false);
var exH=GlissementsExercice(hv,new HashSet<(int,int)>{(0,2),(0,3),(0,5)});
var exV=GlissementsExercice(vv,new HashSet<(int,int)>{(2,3),(3,3),(0,3)});
var attenduH=new HashSet<Voiture>{hv with {C=0},hv with {C=1},hv with {C=3}};
var attenduV=new HashSet<Voiture>{vv with {R=1},vv with {R=3},vv with {R=4}};
if(exH is null||exV is null)Console.WriteLine("Exercice 3 a completer");
else if(exH.Count!=attenduH.Count||exV.Count!=attenduV.Count||!attenduH.SetEquals(exH)||!attenduV.SetEquals(exV))throw new Exception("glissements incorrects");else Console.WriteLine("Exercice 3 : deux ensembles exacts valides");

Exercice 3 a completer


## 6. Benchmark : six variantes distinctes

On parcourt les scrambles de 0 à 38 glissements par pas de 2 et les graines 1 à 19. Après déduplication par clé, seules les instances dont BFS développe au moins 100 états sont retenues, jusqu'à six. Les générateurs aléatoires C# et Python diffèrent : leurs instances et chiffres peuvent différer, mais le protocole et les budgets sont identiques.

In [11]:
var lignes=new List<(string etat,Recherche b,Recherche a,Recherche g)>();var deja=new HashSet<string>();
for(int steps=0;steps<40&&lignes.Count<6;steps+=2)for(int seed=1;seed<20&&lignes.Count<6;seed++){
 var s=Rush.Melanger(instance,steps,seed);if(!deja.Add(s))continue;var b=Rush.BFS(s);if(b.Chemin is null||b.Developpes<100)continue;var a=Rush.AStar(s);var gr=Rush.Glouton(s);
 if(!Rush.CheminValide(s,b)||!Rush.CheminValide(s,a)||b.Chemin.Count!=a.Chemin!.Count||(gr.Chemin is not null&&!Rush.CheminValide(s,gr)))throw new Exception("chemin ou optimum benchmark");lignes.Add((s,b,a,gr));
}
if(lignes.Count!=6||lignes.Select(x=>x.etat).Distinct().Count()!=6)throw new Exception("six instances distinctes requises");
Console.WriteLine("# | BFS etats | A* etats | glouton etats | optimum coups | glouton coups");
for(int i=0;i<lignes.Count;i++){var x=lignes[i];Console.WriteLine($"{i+1} | {x.b.Developpes} | {x.a.Developpes} | {x.g.Developpes} | {x.b.Chemin!.Count-1} | {(x.g.Chemin is null?"cul-de-sac":(x.g.Chemin.Count-1).ToString())}");}
int totalB=lignes.Sum(x=>x.b.Developpes), totalA=lignes.Sum(x=>x.a.Developpes);
Console.WriteLine($"six cles distinctes, seuil BFS >=100 ; totaux BFS={totalB}, A*={totalA}, rapport={(double)totalB/totalA:F2}");

# | BFS etats | A* etats | glouton etats | optimum coups | glouton coups


1 | 433 | 100 | 135 | 3 | 134


2 | 504 | 152 | 134 | 3 | 133


3 | 481 | 146 | 135 | 3 | 134


4 | 117 | 34 | 134 | 2 | 133


5 | 432 | 121 | 135 | 3 | 134


6 | 104 | 31 | 135 | 2 | 134


six cles distinctes, seuil BFS >=100 ; totaux BFS=2071, A*=584, rapport=3.55


Le tableau compare les mêmes instances pour les trois méthodes ; BFS et A* ont été contrôlés pour fournir des chemins légaux de même longueur. Le rapport total décrit cet échantillon : il ne prouve pas qu'A* développe toujours moins d'états ni que le glouton a toujours le même comportement.

Dans cette exécution C#, les six instances totalisent 2 071 états développés par BFS et 584 par A*, soit un rapport de 3,55. Leurs optima valent 2 ou 3 coups ; le glouton en utilise 133 ou 134.

## 7. Exercices — récapitulatif

Les stubs répartis dans les sections précédentes portent sur le but, les bloqueurs et l'énumération des glissements. Ils restent ouverts ; leurs gardes vérifient plusieurs fixtures une fois complétés.

## À retenir

Chaque glissement vaut un coup, quelle que soit sa longueur. BFS trouve un plus court chemin ; A* le retrouve avec l'heuristique admissible des bloqueurs ; le glouton n'a pas cette garantie. Les états développés et la longueur d'une route répondent à deux questions différentes.

## Vérifiez votre compréhension

<details><summary>Pourquoi compter les bloqueurs ne surestime-t-il pas le nombre de coups ?</summary>Chaque voiture qui barre la sortie doit être déplacée au moins une fois.</details>

<details><summary>Le glouton garantit-il un chemin optimal ?</summary>Non : il ne tient pas compte du coût déjà parcouru et ne revient pas en arrière.</details>

## Pour aller plus loin

Comparer avec [App-34 — 15-puzzle A*](App-34-15Puzzle-Admissible-Astar-CSharp.ipynb) et reprendre le benchmark avec d'autres puzzles. Korf (1985), IDA* ; Hider (2003), benchmarks Rush Hour.